Alur ekstraksi fitur:
  1. Pre-Emphasis
  2. Frame Blocking
  3. Windowing (Hamming)
  4. Fast Fourier Transform (FFT)
  5. Mel Filter Bank
  6. Logarithm
  7. Discrete Cosine Transform (DCT)
  8. Deltas and Double Deltas

Formula:
  - Pre-Emphasis  : X(n) = y(n) - a × y(n-1)              [2.2]
  - Hamming Window: w(n) = 0.54 - 0.46 × cos(2πn / (N-1)) [2.3]
  - Mel Scale     : Mel(f) = 1127 × ln(1 + f/700)         [2.4]
  - DCT (MFCC)    : Cn = Σ log(sk) × cos(πn/K × (k-0.5))  [2.5]
  - Delta         : δ(t) = c(t+1) - c(t)                  [2.6]

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
sampled_root = '../data/sampled'
sampled2_root = '../data/sampled2'

In [ ]:
# ─────────────────────────────────────────────
#  Step 1 : Pre-Emphasis
#  Formula [2.2]: X(n) = y(n) - a × y(n-1)
# ─────────────────────────────────────────────
def pre_emphasis(signal : np.ndarray, a = 0.97):
    return np.append(signal[0],signal[1:] - (a* signal[:-1]))

In [ ]:
# ─────────────────────────────────────────────
#  Step 2 : Frame Blocking
# ─────────────────────────────────────────────
def frame_blocking(signal : np.ndarray, sample_rate : int,
                   frame_size, frame_distance ): 
    frame_length =(int)(round(frame_size * sample_rate))
    frame_diff   =(int)(round(frame_distance * sample_rate))
    signal_length = len(signal)
    n_frames = 1 + (signal_length - frame_length) // frame_diff 
    
    indices = np.array([
        signal[i * frame_diff : i * frame_diff + frame_length]
        for i in range(n_frames)
    ])
    
    indices = np.minimum (indices, signal_length - 1 )
    frames  = signal[indices]
    return frames

In [ ]:
# ─────────────────────────────────────────────
#  Step 3 : Windowing (Hamming)
# ─────────────────────────────────────────────
def windowing(frames : np.ndarray) :
    frame_Shape = frames.shapes[1]
    n = np.arange(frame_Shape)
    
    hamming = 0.54 - 0.46 * np.cos * (2 * np.pi * n // frame_Shape - 1)
    windowing  = frames * hamming
    return windowing

In [ ]:
def fast_fourier_transform(windowing : np.ndarray) : 
    #zero_paddding 
     if(windowing)

In [ ]:
# ─────────────────────────────────────────────
#  Step 4 : Fast Fourier Transform (FFT)
# ─────────────────────────────────────────────
def apply_fft(windowed_frames: np.ndarray, n_fft: int = 512) -> np.ndarray:
    """
    Menghitung respons frekuensi magnitudo untuk setiap bingkai
    menggunakan Fast Fourier Transform.

    Args:
        windowed_frames : bingkai setelah windowing
        n_fft           : ukuran FFT (zero-padding jika perlu)

    Returns:
        power_spectrum  : spektrum daya per bingkai
    """
    # FFT → ambil nilai absolut kuadrat sebagai spektrum daya
    magnitude_spectrum = np.abs(np.fft.rfft(windowed_frames, n=n_fft))
    power_spectrum     = (1.0 / n_fft) * (magnitude_spectrum ** 2)
    return power_spectrum


In [ ]:
# ─────────────────────────────────────────────
#  Konfigurasi Default (dapat diubah sesuai kebutuhan)
# ─────────────────────────────────────────────
class MFCCConfig:
    sample_rate:    int   = 16000   # Hz
    pre_emphasis:   float = 0.97    # koefisien a ∈ [0.85, 1]  – paper formula 2.2
    frame_size:     float = 0.025   # detik  → 25 ms
    frame_stride:   float = 0.010   # detik  → 10 ms (overlap)
    n_fft:          int   = 512     # ukuran FFT
    n_mels:         int   = 40      # jumlah filter Mel (K)
    n_mfcc:         int   = 13      # jumlah koefisien MFCC yang diambil
    fmin:           float = 0.0     # frekuensi minimal filter bank
    fmax:           float = None    # None → sample_rate / 2




# ─────────────────────────────────────────────
#  Step 5 : Mel Filter Bank
#  Formula [2.4]: Mel(f) = 1127 × ln(1 + f/700)
# ─────────────────────────────────────────────
def mel_filter_bank(power_spectrum: np.ndarray,
                    sample_rate: int,
                    n_fft: int    = 512,
                    n_mels: int   = 40,
                    fmin: float   = 0.0,
                    fmax: float   = None) -> np.ndarray:
    """
    Menerapkan serangkaian filter bandpass segitiga pada skala Mel
    untuk memperoleh energi logaritma yang sesuai dari setiap filter.

    Args:
        power_spectrum : spektrum daya (num_frames × n_fft//2+1)
        sample_rate    : frekuensi sampling
        n_fft          : ukuran FFT
        n_mels         : jumlah filter Mel (K)
        fmin / fmax    : batas frekuensi filter

    Returns:
        filter_bank_energies : energi per filter per bingkai
    """
    if fmax is None:
        fmax = sample_rate / 2.0

    # Konversi batas frekuensi ke skala Mel: Mel(f) = 1127 * ln(1 + f/700)
    mel_min = 1127.0 * np.log(1.0 + fmin / 700.0)
    mel_max = 1127.0 * np.log(1.0 + fmax / 700.0)

    # K+2 titik Mel yang tersebar merata
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)

    # Balik ke frekuensi Hz: f = 700 * (e^(m/1127) - 1)
    hz_points = 700.0 * (np.exp(mel_points / 1127.0) - 1.0)

    # Ubah frekuensi Hz ke indeks FFT
    bin_points = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)

    # Bangun matriks filter segitiga (n_mels × n_fft//2+1)
    n_freq_bins = n_fft // 2 + 1
    filter_matrix = np.zeros((n_mels, n_freq_bins))
    for m in range(1, n_mels + 1):
        f_left   = bin_points[m - 1]
        f_center = bin_points[m]
        f_right  = bin_points[m + 1]

        # Sisi kiri filter segitiga
        for k in range(f_left, f_center):
            if f_center != f_left:
                filter_matrix[m - 1, k] = (k - f_left) / (f_center - f_left)
        # Sisi kanan filter segitiga
        for k in range(f_center, f_right):
            if f_right != f_center:
                filter_matrix[m - 1, k] = (f_right - k) / (f_right - f_center)

    # Terapkan filter ke spektrum daya → energi setiap filter
    filter_bank_energies = power_spectrum @ filter_matrix.T   # (num_frames × n_mels)
    # Hindari log(0) dengan nilai minimum kecil
    filter_bank_energies = np.maximum(filter_bank_energies, 1e-10)
    return filter_bank_energies


# ─────────────────────────────────────────────
#  Step 6 : Logarithm
# ─────────────────────────────────────────────
def apply_log(filter_bank_energies: np.ndarray) -> np.ndarray:
    """
    Menerapkan logaritma pada energi spektrum Mel karena pendengaran
    manusia bersifat tidak linear (skala logaritmik).

    Args:
        filter_bank_energies : energi filter bank (num_frames × n_mels)

    Returns:
        log_energies : log energi filter bank
    """
    return np.log(filter_bank_energies)


# ─────────────────────────────────────────────
#  Step 7 : Discrete Cosine Transform (DCT)
#  Formula [2.5]: Cn = Σ log(sk) × cos(πn/K × (k - 0.5))
# ─────────────────────────────────────────────
def apply_dct(log_energies: np.ndarray, n_mfcc: int = 13) -> np.ndarray:
    """
    Menerapkan DCT untuk mendekorelasi energi bank filter yang tumpang
    tindih dan menghasilkan koefisien MFCC.

    Formula [2.5]:
        Cn = Σ_{k=1}^{K} log(sk) × cos(πn/K × (k - 0.5))

    Args:
        log_energies : log energi filter bank (num_frames × K)
        n_mfcc       : jumlah koefisien MFCC yang diambil (n)

    Returns:
        mfcc : koefisien MFCC (num_frames × n_mfcc)
    """
    K      = log_energies.shape[1]               # total filter Mel
    n_idx  = np.arange(n_mfcc)                   # indeks koefisien  0 .. n_mfcc-1
    k_idx  = np.arange(1, K + 1)                 # indeks filter     1 .. K

    # Matriks kosinus: (n_mfcc × K)
    # cos(π * n / K * (k - 0.5)) → sesuai formula paper dengan penyesuaian indeks
    cos_matrix = np.cos(
        np.pi * n_idx[:, np.newaxis] / K * (k_idx[np.newaxis, :] - 0.5)
    )

    # Cn = Σ log(sk) * cos(...)  → (num_frames × n_mfcc)
    mfcc = log_energies @ cos_matrix.T
    return mfcc


# ─────────────────────────────────────────────
#  Step 8 : Deltas and Double Deltas
#  Formula [2.6]: δ(t) = c(t+1) - c(t)
# ─────────────────────────────────────────────
def compute_deltas(mfcc: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Menghitung koefisien delta dan double delta untuk menangkap
    perubahan fitur antar bingkai waktu (dinamika temporal).

    Formula [2.6]:
        δ(t) = c(t+1) - c(t)

    Args:
        mfcc : koefisien MFCC statis (num_frames × n_mfcc)

    Returns:
        delta        : koefisien delta  δ(t)
        double_delta : koefisien double delta  δ(δ(t))
    """
    # Delta: selisih bingkai berurutan
    delta        = np.diff(mfcc,        axis=0, prepend=mfcc[:1])
    # Double delta: selisih dari delta
    double_delta = np.diff(delta,       axis=0, prepend=delta[:1])
    return delta, double_delta


# ─────────────────────────────────────────────
#  Pipeline Utama
# ─────────────────────────────────────────────
def extract_mfcc(signal: np.ndarray,
                 config: MFCCConfig = None) -> dict:
    """
    Menjalankan seluruh pipeline ekstraksi fitur MFCC sesuai paper.

    Args:
        signal : array sinyal audio (mono, float32/float64)
        config : objek konfigurasi MFCCConfig

    Returns:
        result : dict berisi output setiap step dan fitur gabungan
    """
    if config is None:
        config = MFCCConfig()

    # ── Step 1: Pre-Emphasis ─────────────────
    emphasized = pre_emphasis(signal, a=config.pre_emphasis)

    # ── Step 2: Frame Blocking ───────────────
    frames = frame_blocking(
        emphasized,
        config.sample_rate,
        config.frame_size,
        config.frame_stride
    )

    # ── Step 3: Windowing (Hamming) ──────────
    windowed = windowing(frames)

    # ── Step 4: FFT ──────────────────────────
    power_spec = apply_fft(windowed, n_fft=config.n_fft)

    # ── Step 5: Mel Filter Bank ──────────────
    filter_energies = mel_filter_bank(
        power_spec,
        config.sample_rate,
        n_fft=config.n_fft,
        n_mels=config.n_mels,
        fmin=config.fmin,
        fmax=config.fmax
    )

    # ── Step 6: Logarithm ────────────────────
    log_energies = apply_log(filter_energies)

    # ── Step 7: DCT → Koefisien MFCC ─────────
    mfcc = apply_dct(log_energies, n_mfcc=config.n_mfcc)

    # ── Step 8: Delta & Double Delta ─────────
    delta, double_delta = compute_deltas(mfcc)

    # Gabungkan: [MFCC | Δ | ΔΔ] → vektor fitur lengkap
    features = np.concatenate([mfcc, delta, double_delta], axis=1)

    return {
        "emphasized"       : emphasized,
        "frames"           : frames,
        "windowed"         : windowed,
        "power_spectrum"   : power_spec,
        "filter_energies"  : filter_energies,
        "log_energies"     : log_energies,
        "mfcc"             : mfcc,
        "delta"            : delta,
        "double_delta"     : double_delta,
        "features"         : features,        # fitur lengkap [MFCC | Δ | ΔΔ]
    }


# ─────────────────────────────────────────────
#  Visualisasi Setiap Step
# ─────────────────────────────────────────────
def visualize_pipeline(signal: np.ndarray,
                       result: dict,
                       config: MFCCConfig,
                       save_path: str = None):
    """Menampilkan visualisasi setiap tahap pipeline MFCC."""
    t = np.arange(len(signal)) / config.sample_rate

    fig = plt.figure(figsize=(16, 18))
    fig.suptitle("Pipeline Ekstraksi Fitur MFCC\n(Perbandingan GMM & TDNN – Pengenalan Suara Bahasa Jawa)",
                 fontsize=14, fontweight="bold", y=0.98)

    gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.55, wspace=0.35)

    # 1. Sinyal asli & setelah Pre-Emphasis
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t, signal, color="#4C72B0", linewidth=0.6, label="Asli")
    ax1.plot(t, result["emphasized"], color="#DD8452", linewidth=0.6,
             alpha=0.8, label="Pre-Emphasis")
    ax1.set_title("Step 1 – Pre-Emphasis\nX(n) = y(n) − a × y(n−1)", fontsize=9)
    ax1.set_xlabel("Waktu (s)"); ax1.set_ylabel("Amplitudo")
    ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

    # 2. Frame Blocking
    ax2 = fig.add_subplot(gs[0, 1])
    n_show = min(5, result["frames"].shape[0])
    for i in range(n_show):
        ax2.plot(result["frames"][i] + i * 0.3, linewidth=0.7,
                 color=plt.cm.tab10(i / n_show))
    ax2.set_title(f"Step 2 – Frame Blocking\n({result['frames'].shape[0]} bingkai)", fontsize=9)
    ax2.set_xlabel("Sampel"); ax2.set_ylabel("Amplitudo (offset)")
    ax2.grid(alpha=0.3)

    # 3. Windowing (Hamming)
    ax3 = fig.add_subplot(gs[1, 0])
    N    = result["frames"].shape[1]
    n_   = np.arange(N)
    ham  = 0.54 - 0.46 * np.cos(2 * np.pi * n_ / (N - 1))
    ax3.fill_between(n_, ham, alpha=0.4, color="#55A868")
    ax3.plot(n_, ham, color="#55A868", linewidth=1.2)
    ax3.set_title("Step 3 – Windowing (Hamming)\nw(n) = 0.54 − 0.46·cos(2πn/(N−1))", fontsize=9)
    ax3.set_xlabel("Sampel"); ax3.set_ylabel("Bobot"); ax3.grid(alpha=0.3)

    # 4. Spektrum Daya (FFT)
    ax4 = fig.add_subplot(gs[1, 1])
    freq_axis = np.linspace(0, config.sample_rate / 2,
                            result["power_spectrum"].shape[1])
    avg_power = result["power_spectrum"].mean(axis=0)
    ax4.semilogy(freq_axis, avg_power + 1e-10, color="#C44E52", linewidth=0.8)
    ax4.set_title("Step 4 – FFT (rata-rata spektrum daya)", fontsize=9)
    ax4.set_xlabel("Frekuensi (Hz)"); ax4.set_ylabel("Daya (log)")
    ax4.grid(alpha=0.3)

    # 5. Mel Filter Bank energies
    ax5 = fig.add_subplot(gs[2, 0])
    im5 = ax5.imshow(result["filter_energies"].T, aspect="auto", origin="lower",
                     cmap="magma")
    ax5.set_title("Step 5 – Mel Filter Bank Energies", fontsize=9)
    ax5.set_xlabel("Bingkai"); ax5.set_ylabel("Filter Mel")
    fig.colorbar(im5, ax=ax5, pad=0.02)

    # 6. Log Mel Energies
    ax6 = fig.add_subplot(gs[2, 1])
    im6 = ax6.imshow(result["log_energies"].T, aspect="auto", origin="lower",
                     cmap="viridis")
    ax6.set_title("Step 6 – Log Mel Energies", fontsize=9)
    ax6.set_xlabel("Bingkai"); ax6.set_ylabel("Filter Mel")
    fig.colorbar(im6, ax=ax6, pad=0.02)

    # 7. Koefisien MFCC (DCT)
    ax7 = fig.add_subplot(gs[3, 0])
    im7 = ax7.imshow(result["mfcc"].T, aspect="auto", origin="lower",
                     cmap="coolwarm")
    ax7.set_title("Step 7 – DCT → Koefisien MFCC\nCn = Σ log(sk)·cos(πn/K·(k−0.5))",
                  fontsize=9)
    ax7.set_xlabel("Bingkai"); ax7.set_ylabel("Koefisien MFCC")
    fig.colorbar(im7, ax=ax7, pad=0.02)

    # 8. Delta & Double Delta
    ax8 = fig.add_subplot(gs[3, 1])
    im8 = ax8.imshow(result["delta"].T, aspect="auto", origin="lower",
                     cmap="PuOr")
    ax8.set_title("Step 8 – Delta (δ(t) = c(t+1) − c(t))", fontsize=9)
    ax8.set_xlabel("Bingkai"); ax8.set_ylabel("Δ MFCC")
    fig.colorbar(im8, ax=ax8, pad=0.02)

    # 9. Fitur Gabungan [MFCC | Δ | ΔΔ]
    ax9 = fig.add_subplot(gs[4, :])
    im9 = ax9.imshow(result["features"].T, aspect="auto", origin="lower",
                     cmap="inferno")
    ax9.set_title(
        f"Fitur Gabungan [MFCC | Δ | ΔΔ]  –  {result['features'].shape[1]} dimensi "
        f"({result['features'].shape[0]} bingkai)",
        fontsize=9
    )
    ax9.set_xlabel("Bingkai (waktu)"); ax9.set_ylabel("Dimensi Fitur")
    fig.colorbar(im9, ax=ax9, pad=0.01)

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[✓] Visualisasi disimpan: {save_path}")
    plt.show()


# ─────────────────────────────────────────────
#  Contoh Penggunaan
# ─────────────────────────────────────────────
def demo_with_synthetic():
    """
    Demo menggunakan sinyal sintetis (tanpa file audio).
    Ganti bagian ini dengan librosa.load() untuk file nyata.
    """
    config = MFCCConfig()
    np.random.seed(42)
    duration  = 2.0   # detik
    n_samples = int(config.sample_rate * duration)

    # Sinyal sintetis: campuran sinusoidal + noise (menyerupai ucapan)
    t      = np.linspace(0, duration, n_samples)
    signal = (
        0.5 * np.sin(2 * np.pi * 200  * t) +
        0.3 * np.sin(2 * np.pi * 400  * t) +
        0.2 * np.sin(2 * np.pi * 800  * t) +
        0.05 * np.random.randn(n_samples)
    ).astype(np.float32)

    print("=" * 60)
    print("Pipeline Ekstraksi Fitur MFCC")
    print("Referensi: Section 2.2.8 – GMM & TDNN Bahasa Jawa")
    print("=" * 60)
    print(f"  Sinyal input       : {n_samples} sampel @ {config.sample_rate} Hz")

    result = extract_mfcc(signal, config)

    print(f"\n  Step 1 Pre-Emphasis   : {result['emphasized'].shape}")
    print(f"  Step 2 Frame Blocking : {result['frames'].shape}  "
          f"({result['frames'].shape[0]} bingkai × {result['frames'].shape[1]} sampel)")
    print(f"  Step 3 Windowing      : {result['windowed'].shape}")
    print(f"  Step 4 FFT            : {result['power_spectrum'].shape}")
    print(f"  Step 5 Mel Filter Bank: {result['filter_energies'].shape}  "
          f"(K={config.n_mels} filter)")
    print(f"  Step 6 Log            : {result['log_energies'].shape}")
    print(f"  Step 7 DCT (MFCC)     : {result['mfcc'].shape}  "
          f"(n={config.n_mfcc} koefisien)")
    print(f"  Step 8 Delta          : {result['delta'].shape}")
    print(f"         Double Delta   : {result['double_delta'].shape}")
    print(f"\n  ► Fitur Gabungan [MFCC|Δ|ΔΔ]: {result['features'].shape}")
    print("=" * 60)

    visualize_pipeline(signal, result, config,
                       save_path="/mnt/user-data/outputs/mfcc_pipeline_visualization.png")
    return result


def load_and_extract(audio_path: str, config: MFCCConfig = None) -> dict:
    """
    Memuat file audio dan menjalankan pipeline MFCC.

    Args:
        audio_path : path ke file audio (.wav, .mp3, dll.)
        config     : konfigurasi (default jika None)

    Returns:
        result : dict hasil ekstraksi fitur
    """
    if config is None:
        config = MFCCConfig()

    signal, sr = librosa.load(audio_path, sr=config.sample_rate, mono=True)
    print(f"[✓] Audio dimuat: {audio_path}  ({len(signal)} sampel @ {sr} Hz)")
    return extract_mfcc(signal, config)


# ─────────────────────────────────────────────
if __name__ == "__main__":
    result = demo_with_synthetic()